# What is Quantization?

Your model weights are normally stored as FP32, so decimals with 32 bits

weight = 0.34521847

Quantization converts that to something like FP16 or INT8 (8-bit integer) to make it easy on the CPU

weight = 35

There are three types of quantization:

1. Post-Training Quantization (PTQ) - easiest
    - Train --> quantize after 
    - No retraining needed
    - Slight accuracy drop

2. Quantization-Aware Training (QAT) - better accuracy 
    - **Simulate** quantization DURING training
    - model learns to handle reduced precision
    - Better accuracy than PTQ

3. Dynamic Quantization - the middle ground 
    - weights are quantized, and activations get quantized during runtime 
    - Good for NLP models like BERT or SSR

SO here's the thing, FP32 --> INT8 is most common because integer math is a lot nicer than decimal math. ROCm uses quantization

# Hour 1: Hands-On Quantization 

we are gonna do Dynamic Quantization today

step 1: load housing regression model from before 

In [1]:
import torch 
import torch.nn as nn

#load model structure as before
class LinearRegression(nn.Module):

    def __init__(self,in_features):

        super().__init__()
        self.in_features = in_features

        #formula: linear --> batchnorm1d --> relu --> dropout
        self.linear_stack = nn.Sequential(
            nn.Linear(self.in_features,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.fc = nn.Linear(32,1)

    def forward(self,x):

        x = self.linear_stack(x)
        x = self.fc(x)
        return x
    
model = LinearRegression(in_features=8)
model.load_state_dict(torch.load("best_housing_reg.pth",weights_only=True))
model.eval()
print("Loaded housing model")

Loaded housing model


step 2: applying **dynamic quantization**

Need to run on CPU for inference
- Train on GPU
- Quantize on CPU
- Deploy quantized model on target hardware

In [2]:
import torch.quantization
import copy

# Set quantization engine for non-CUDA devices
torch.backends.quantized.engine = 'qnnpack'  #Works on ARM/Apple Silicon

# #go back to CPU
# model_cpu = copy.deepcopy(model).cpu()
# model_cpu.eval()

quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

print("✓ Model quantized!")

✓ Model quantized!


step 3: compare model sizes

In [3]:
import os 

torch.save(model.state_dict(),"original_model.pth")
torch.save(quantized_model.state_dict(),"quantized_model.pth")

original_size = os.path.getsize("original_model.pth") / 1024
quantized_size = os.path.getsize("quantized_model.pth") / 1024

print(f"Original:  {original_size:.2f} KB")
print(f"Quantized: {quantized_size:.2f} KB")
print(f"Reduction: {original_size/quantized_size:.2f}x smaller!")

Original:  27.14 KB
Quantized: 16.95 KB
Reduction: 1.60x smaller!


step 4 : compare speed 

In [4]:
import time

test_input = torch.randn(1000, 8)

# Original speed
start = time.time()
with torch.no_grad():
    for _ in range(100):
        _ = model(test_input)
original_time = time.time() - start

# Quantized speed
start = time.time()
with torch.no_grad():
    for _ in range(100):
        _ = quantized_model(test_input)
quantized_time = time.time() - start

print(f"Original:  {original_time:.4f}s")
print(f"Quantized: {quantized_time:.4f}s")
print(f"Speedup:   {original_time/quantized_time:.2f}x faster!")

Original:  0.0243s
Quantized: 0.0449s
Speedup:   0.54x faster!


[W618 18:16:35.407148000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


step 5: verify accuracy didn't drop much

In [5]:
with torch.no_grad():
    orig_pred = model(test_input)
    quant_pred = quantized_model(test_input)

diff = torch.abs(orig_pred - quant_pred).mean()
print(f"Average prediction difference: {diff:.6f}")
# Should be tiny (< 0.01)

Average prediction difference: 0.021101


So our quantized model was slower than the orignal. On smaller models, the overhead can outweight the benefits. This is why dynamic quantization is better for really large models like BERT. 